In [1]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

def load_chroma_db():

    embeddings = OllamaEmbeddings(
        model="Qwen3-Embedding-0.6B-Q8_0:latest",
        temperature=0,
    )

    vector_store = Chroma(
        embedding_function = embeddings,
        collection_name = "chat_documents_instagram",
        persist_directory='.chroma_db',
    )
    return vector_store
vector_store = load_chroma_db()

In [2]:
collection = vector_store._collection
print(f"Number of documents: {collection.count()}")

Number of documents: 4204


### Instantiate Generation Model

In [3]:
from langchain_ollama import ChatOllama

chat_model = ChatOllama(
    model="qwen2.5:7b-instruct",
    num_predict=256,
    num_ctx=4096,          # Smaller context window
    system="You are a WhatsApp chat analysis agent that retrieves and analyzes conversation data to answer user questions about their chat history."
)

#### Setup RAG components

In [4]:
from typing_extensions import List, TypedDict, Annotated
from typing import Optional, Literal

class WhatsAppSearch(TypedDict):
    """WhatsApp chat search query with filtering."""
    participants: Annotated[
        Optional[List[str]],
        ...,
        "If there are no names of a specific person in the query leave the field, empty."
        "Name of person if mentioned in the question. " \
        
    ]
    time_period: Annotated[
        Optional[Literal["day", "month", "year"]],
        ...,
        "Type of time period: day, month, year, leave empty if no time period mentioned."
    ]
    day: Annotated[Optional[int], ..., "Day number (1-31). Only set for day-specific queries."]
    month: Annotated[Optional[int], ..., "Month number (1-12). Set for day and month queries."]
    year: Annotated[Optional[int], ..., "Year number (e.g. 2023). Set for all time queries."]
    time_query: Annotated[
        Optional[Literal["within", "before", "after"]],
        ...,
        "Time relationship: 'within' = DURING (e.g. 'conversations 2024', 'in March', 'from 2023', 'during June'), 'before' = PRIOR TO (e.g. 'before 2024', 'prior to March'), 'after' = FOLLOWING (e.g. 'after 2024', 'since March'). Leave empty if no time relationship mentioned."
    ]


def analyze_query(question: str) -> WhatsAppSearch:
    """Analyze user question with separate date fields."""
    structured_llm = chat_model.with_structured_output(schema=WhatsAppSearch)
    analysis_prompt = (
        "If present, Extract from this WhatsApp question:\n"
        "- Person names mentioned, i\n"
        "- Time references and whether it's before/after/within a time period\n"
        "- Extract date components separately:\n"
        "  * 'March 2023' → month: 3, year: 2023\n"
        "  * '2023' → year: 2023\n"
        "  * 'June 15, 2024' → day: 15, month: 6, year: 2024\n"
        " Do NOT hallucinate names or dates if they are not there."
        f"Question: {question}"
    )
    query_analysis = structured_llm.invoke(analysis_prompt)
    query_analysis['query'] = question
    return query_analysis


In [5]:
from datetime import datetime

def process_date_query(search_params):
    """Process date query parameters using separate fields."""
    try:
        time_period = search_params.get('time_period')
        time_query = search_params.get('time_query')
        day = search_params.get('day')
        month = search_params.get('month') 
        year = search_params.get('year')
        
        if not all([time_period, time_query, year]):
            return None
        
        if time_period == 'day' and day and month:
            target_date = datetime(year, month, day).date()
            return {
                'type': 'day',
                'date': target_date,
                'time_query': time_query
            }
        elif time_period == 'month' and month:
            return {
                'type': 'month',
                'month': month,
                'year': year,
                'time_query': time_query
            }
        elif time_period == 'year':
            return {
                'type': 'year',
                'year': year,
                'time_query': time_query
            }
    except ValueError as e:
        print(f'Could not load date: {e}')
        return None
    

def date_filter_logic(doc_start_date, doc_end_date, date_criteria):
    """Apply date filtering logic based on criteria."""
    if not date_criteria:
        return True
    
    filter_type = date_criteria['type']
    time_query = date_criteria['time_query']
    
    if filter_type == 'day':
        target_date = date_criteria['date']
        
        if time_query == 'within':
            # Check if target date falls within document's date range
            return doc_start_date <= target_date <= doc_end_date
        elif time_query == 'before':
            # Document must end before target date
            return doc_end_date < target_date
        elif time_query == 'after':
            # Document must start after target date
            return doc_start_date > target_date
    
    elif filter_type == 'month':
        month = date_criteria['month']
        year = date_criteria['year']
        
        if time_query == 'within':
            # Check if any part of document overlaps with the target month
            doc_start_month = (doc_start_date.year, doc_start_date.month)
            doc_end_month = (doc_end_date.year, doc_end_date.month)
            target_month = (year, month)
            
            return doc_start_month <= target_month <= doc_end_month
        elif time_query == 'before':
            # Document must end before the target month
            return (doc_end_date.year, doc_end_date.month) < (year, month)
        elif time_query == 'after':
            # Document must start after the target month
            return (doc_start_date.year, doc_start_date.month) > (year, month)
    
    elif filter_type == 'year':
        year = date_criteria['year']
        
        if time_query == 'within':
            # Check if any part of document overlaps with the target year
            return doc_start_date.year <= year <= doc_end_date.year
        elif time_query == 'before':
            # Document must end before the target year
            return doc_end_date.year < year
        elif time_query == 'after':
            # Document must start after the target year
            return doc_start_date.year > year
    
    return True

In [6]:
# Fix your date_filter_logic function first:
from datetime import datetime
import ast
from langchain_core.tools import tool

USE_PRE_FILTERING = False  # Set to False for post-filtering


def date_filter_logic(date_criteria):
    if not date_criteria:
        return {}
        
    filter_type = date_criteria['type']
    time_query = date_criteria['time_query']
    
    if filter_type == 'day':
        target_date = int(date_criteria['date'].strftime("%Y%m%d"))
        if time_query == 'before':
            return {'end_date': {'$lt': target_date}}  
        elif time_query == 'after':
            return {'start_date': {'$gt': target_date}}  
        elif time_query == 'within':
            return {"$and": [
                {"start_date": {"$lte": target_date}},  
                {"end_date": {"$gte": target_date}},    
            ]}
    
    elif filter_type == 'month':
        month = date_criteria['month']
        year = date_criteria['year']
        start_int = int(f"{year}{month:02d}01")
        # Get last day of month
        if month == 12:
            end_int = int(f"{year}{month:02d}31")
        else:
            next_month = datetime(year, month + 1, 1) 
            end_int = int(next_month.strftime("%Y%m%d"))
        
        if time_query == 'before':
            return {'end_date': {'$lt': start_int}}
        elif time_query == 'after':
            return {'start_date': {'$gte': end_int}}
        elif time_query == 'within':
            return {"$and": [
                {"start_date": {"$lte": end_int}},
                {"end_date": {"$gte": start_int}},
            ]}
    
    elif filter_type == 'year':
        year = date_criteria['year']
        start_int = int(f"{year}0101")
        end_int = int(f"{year}1231")
        
        if time_query == 'before':
            return {'end_date': {'$lt': start_int}}
        elif time_query == 'after':
            return {'start_date': {'$gt': end_int}}
        elif time_query == 'within':
            return {"$and": [
                {"start_date": {"$lte": end_int}},
                {"end_date": {"$gte": start_int}},
            ]}
    
    return {}

def participant_filter_logic(search_params):
    """Generate participant filter for ChromaDB."""
    if not search_params.get('participants'):
        return {}
    
    # For multiple participants, use $or
    if len(search_params['participants']) == 1:
        return {"other_person": search_params['participants'][0]}
    else:
        return {"$or": [{"other_person": p} for p in search_params['participants']]}

# Modified retrieve function - replace retrieve_date with this:
def retrieve_with_prefiltering(date_criteria, search_params):
    """Use pre-filtering instead of post-filtering."""
    
    # Build ChromaDB filter
    filters = []
    
    # Add date filter
    date_filter = date_filter_logic(date_criteria)
    if date_filter:
        filters.append(date_filter)
    
    # Add participant filter  
    participant_filter = participant_filter_logic(search_params)
    if participant_filter:
        filters.append(participant_filter)
    
    # Combine filters
    if len(filters) == 0:
        chroma_filter = None
    elif len(filters) == 1:
        chroma_filter = filters[0]
    else:
        chroma_filter = {"$and": filters}
    
    # Use pre-filtering with ChromaDB
    retrieved_docs = vector_store.similarity_search(
        search_params['query'],
        k=5,  # Can use smaller k since pre-filtered
        filter=chroma_filter
    )
    
    return retrieved_docs
def retrieve_with_postfiltering(date_criteria, search_params):
    """Post-filtering approach - fetch docs then filter with exact same logic as pre-filtering."""
    # Get more documents to filter from
    all_docs = vector_store.similarity_search(
        search_params['query'],
        k=25,  # Get more docs to filter from
    )
    
    filtered_docs = []
    for doc in all_docs:
        metadata = doc.metadata
        include_doc = True
        
        # Apply participant filtering - exact same logic as pre-filtering
        if search_params.get('participants'):
            other_person = metadata.get('other_person', '')
            if len(search_params['participants']) == 1:
                # Single participant: exact match
                include_doc = other_person == search_params['participants'][0]
            else:
                # Multiple participants: any match
                include_doc = other_person in search_params['participants']
        
        # Apply date filtering - exact same logic as pre-filtering
        if include_doc and date_criteria:
            start_date_int = metadata.get('start_date')
            end_date_int = metadata.get('end_date')
            
            if start_date_int is None or end_date_int is None:
                include_doc = False
            else:
                filter_type = date_criteria['type']
                time_query = date_criteria['time_query']
                
                if filter_type == 'day':
                    target_date = int(date_criteria['date'].strftime("%Y%m%d"))
                    if time_query == 'before':
                        include_doc = end_date_int < target_date
                    elif time_query == 'after':
                        include_doc = start_date_int > target_date
                    elif time_query == 'within':
                        include_doc = start_date_int <= target_date and end_date_int >= target_date
                
                elif filter_type == 'month':
                    month = date_criteria['month']
                    year = date_criteria['year']
                    start_int = int(f"{year}{month:02d}01")
                    # Use exact same month end calculation as pre-filtering
                    if month == 12:
                        end_int = int(f"{year}{month:02d}31")
                    else:
                        next_month = datetime(year, month + 1, 1)
                        end_int = int(next_month.strftime("%Y%m%d"))
                    
                    if time_query == 'before':
                        include_doc = end_date_int < start_int
                    elif time_query == 'after':
                        include_doc = start_date_int >= end_int
                    elif time_query == 'within':
                        include_doc = start_date_int <= end_int and end_date_int >= start_int
                
                elif filter_type == 'year':
                    year = date_criteria['year']
                    start_int = int(f"{year}0101")
                    end_int = int(f"{year}1231")
                    
                    if time_query == 'before':
                        include_doc = end_date_int < start_int
                    elif time_query == 'after':
                        include_doc = start_date_int > end_int
                    elif time_query == 'within':
                        include_doc = start_date_int <= end_int and end_date_int >= start_int
        
        if include_doc:
            filtered_docs.append(doc)
            if len(filtered_docs) >= 5:
                break
    
    return filtered_docs


# Modified retrieve tool with boolean toggle
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query with intelligent filtering."""
    print(f"\n🔍 RETRIEVE DEBUG - Query: '{query}'")
    
    search_params = analyze_query(query)
    print(f"📊 Search params: {search_params}")
    
    date_criteria = process_date_query(search_params)
    print(f'Date Criteria {date_criteria}')
    
    # Check if filtering is needed
    needs_filtering = date_criteria or search_params.get('participants')
    
    if needs_filtering:
        if USE_PRE_FILTERING:
            print("Using PRE-FILTERING approach")
            retrieved_docs = retrieve_with_prefiltering(date_criteria, search_params)
        else:
            print("Using POST-FILTERING approach")
            retrieved_docs = retrieve_with_postfiltering(date_criteria, search_params)
    else:
        # No filtering needed
        retrieved_docs = vector_store.similarity_search(
            search_params['query'],
            k=5,
        )
    
    print(f'Number of chunks retrieved: {len(retrieved_docs)}')
    
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('chat_name', 'Unknown')}\n"
        f"Date Range: {doc.metadata.get('date_range', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )
    
    return serialized, retrieved_docs

In [16]:
# import ast
# from langchain_core.tools import tool

# def parse_metadata_date_range(date_range_str: str):
#     """Parse metadata date_range string like '2023-01-15 - 2023-01-20' into start and end dates."""
#     try:
#         start_str, end_str = date_range_str.split(' - ')
#         start_date = datetime.strptime(start_str.strip(), '%Y-%m-%d').date()
#         end_date = datetime.strptime(end_str.strip(), '%Y-%m-%d').date()
#         return start_date, end_date
#     except:
#         return None, None


# def extract_date_from_string(date_string):
#     """Extract first valid date pattern from string and return yyyy/MM/dd format."""
#     # Multiple date patterns to try
#     patterns = [
#         r'(\d{4})/(\d{1,2})/(\d{1,2})',  # yyyy/MM/dd or yyyy/M/d
#         r'(\d{1,2})/(\d{1,2})/(\d{4})',  # dd/MM/yyyy or d/M/yyyy
#         r'(\d{4})-(\d{1,2})-(\d{1,2})',  # yyyy-MM-dd
#         r'(\d{1,2})-(\d{1,2})-(\d{4})',  # dd-MM-yyyy
#         r'(\d{4})\.(\d{1,2})\.(\d{1,2})', # yyyy.MM.dd
#         r'(\d{1,2})\.(\d{1,2})\.(\d{4})', # dd.MM.yyyy
#     ]
    
#     for pattern in patterns:
#         match = re.search(pattern, str(date_string))
#         if match:
#             parts = match.groups()
#             # Determine if it's yyyy/MM/dd or dd/MM/yyyy format
#             if len(parts[0]) == 4:  # First part is year
#                 return f"{parts[0]}/{parts[1].zfill(2)}/{parts[2].zfill(2)}"
#             else:  # First part is day
#                 return f"{parts[2]}/{parts[1].zfill(2)}/{parts[0].zfill(2)}"
#     return None


# def retrieve_date(date_criteria,search_params):
    
#     # First retrieve all documents with similarity search
#     all_docs = vector_store.similarity_search(
#         search_params['query'],
#         k=25,  # Get more docs to filter from
#     )
#     # Apply custom filtering
#     filtered_docs = []
#     for doc in all_docs:
#         metadata = doc.metadata
        
#         # Filter by participants if specified
#         participant_match = True
#         if search_params.get('participants'):
#             doc_participants = metadata.get('participants', [])
#             doc_participants = ast.literal_eval(doc_participants)

#             participant_match = any(p.lower() in [dp.lower() for dp in doc_participants]
#                                   for p in search_params['participants'])
#             if not participant_match:
#                 continue

#         # Get document date range
#         doc_date_range = metadata.get('date_range')
#         if not doc_date_range and date_criteria:
#             continue
            
#         date_match = True
#         if date_criteria and doc_date_range:
#             doc_start_date, doc_end_date = parse_metadata_date_range(doc_date_range)
#             if not doc_start_date or not doc_end_date:
#                 continue
            
#             # Apply date filtering logic
#             date_match = date_filter_logic(doc_start_date, doc_end_date, date_criteria)
        
#         if participant_match and date_match:
#             filtered_docs.append(doc)

#         if len(filtered_docs) >= 4:
#             break
    
#     return filtered_docs
    
# @tool(response_format="content_and_artifact")
# def retrieve(query: str):
#     """Retrieve information related to a query with intelligent filtering."""
#     print(f"\n🔍 RETRIEVE DEBUG - Query: '{query}'")
    
#     # Analyze the query for filtering parameters
#     search_params = analyze_query(query)
#     print(f"📊 Search params: {search_params}")
    
#     # Process date query parameters
    
#     date_criteria = process_date_query(search_params)
#     print(f'Date Criteria {date_criteria}')
#     if date_criteria:
#         retrieved_docs = retrieve_date(date_criteria,search_params)
#     else:
#         # First retrieve all documents with similarity search
#         retrieved_docs = vector_store.similarity_search(
#         search_params['query'],
#         k=5,
#         )

#     print(f'Number of chunks retrieved : {len(retrieved_docs)}')    
#     serialized = "\n\n".join(
#     f"Source: {doc.metadata.get('chat_name', 'Unknown')}\n"
#     f"Date Range: {doc.metadata.get('date_range', 'Unknown')}\n"
#     f"Content: {doc.page_content}"
#     for doc in retrieved_docs
#     )
    
#     return serialized, retrieved_docs


In [ ]:
from langchain_core.documents import Document
from typing_extensions import List,TypedDict
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState, StateGraph

### Setting up orchestration of our Steps (retrieval and generation) using LangGraph
class State(TypedDict):
    question : str
    context : List[Document]
    answer: str

def query_or_respond(state: MessagesState):
    """ Generate tool call for retrieval or direct respond."""

    llm_with_tools = chat_model.bind_tools([retrieve])
    system_msg = SystemMessage(
        "You are a helpful assistant. When you don't know something, or are asked something about the user's chats or personal information, "
        "You MUST use the retrieve tool to find relevant information, which returns chunks of relevant WhatsApp Chat History. "
        "Use the retrieve tool for any question you cannot answer with high confidence."
        "When using the retrieve tool, make sure the query parameter is sufficiently detailed to refelct the original question, do not miss any details."
    )
    messages_with_system = [system_msg] + state["messages"]
    response = llm_with_tools.invoke(messages_with_system)
    print(f'State messages {response}')

    return {"messages": [response]}

### declare the tool, so that we can add it in a 'callable' node
tools = ToolNode([retrieve])

def generate(state: MessagesState):
    recent_tool_messages = []

    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]
    
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    
    system_message_content = (
    "You are an assistant for question-answering tasks about the user's past WhatsApp activities and interactions. "
    "Key points:\n"
    "- Use the retrieved chat excerpts to answer the question\n"
    "- Be specific about who said what and when, referencing chat metadata (names, dates, participants)\n"
    "- Focus on messages relevant to the question - multiple topics may appear in the same conversation\n"
    "- If context is unrelated or unclear, respond with 'I don't know about...'. Do NOT Make up facts\n"
    "- Keep answers simple, conversational, and don't directly refer to 'the conversations provided'\n"
    "- Use three sentences maximum\n"
    "- If context is not present, state that there is no information for that request."
    "- If there are no excerpts with 'Me' then assume I did not take part in the discussion."
    "\n\n"
    f"{docs_content}"
    )

    conversation = [
        message
        for message in state["messages"]
        if message.type in ("human","system")
        or (message.type =="ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation   
    response = chat_model.invoke(prompt) 
    return {"messages" : [response]}

In [8]:
graph_builder = StateGraph(MessagesState)

In [9]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition
graph_builder = StateGraph(MessagesState)
graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges( 
    "query_or_respond",
    tools_condition,
    {END: END, "tools" : "tools"}
)
graph_builder.add_edge("tools","generate")
graph_builder.add_edge("generate",END)

graph = graph_builder.compile()

In [10]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
# specify thread_id
config = {"configurable" : {"thread_id" : "abc123"}}

#### Query Analysis - Tests

In [11]:
# year after
input_message = "conversations Cristina 2024"
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

conversations Cristina 2024
State messages content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b-instruct', 'created_at': '2025-08-08T13:54:09.506023Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3247064500, 'load_duration': 665456875, 'prompt_eval_count': 234, 'prompt_eval_duration': 1282133875, 'eval_count': 28, 'eval_duration': 1296295542, 'model_name': 'qwen2.5:7b-instruct'} id='run--cc59e466-984d-46c2-9bfa-895990cbdfb3-0' tool_calls=[{'name': 'retrieve', 'args': {'query': 'conversations with Cristina in 2024'}, 'id': 'd3d71072-9e8e-471c-8071-4b2238a21877', 'type': 'tool_call'}] usage_metadata={'input_tokens': 234, 'output_tokens': 28, 'total_tokens': 262}
================================== Ai Message ==================================
Tool Calls:
  retrieve (d3d71072-9e8e-471c-8071-4b2238a21877)
 Call ID: d3d71072-9e8e-471c-8071-4b2238a21877
  Args:
    query: conversati

In [203]:
# month witihn query
input_message = "Who did I talk during the month of March in 2023?"
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Who did I talk during the month of March in 2023?
State messages content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b-instruct', 'created_at': '2025-08-07T19:27:13.618075Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20241117833, 'load_duration': 18054500, 'prompt_eval_count': 3122, 'prompt_eval_duration': 18560697125, 'eval_count': 29, 'eval_duration': 1645306916, 'model_name': 'qwen2.5:7b-instruct'} id='run--e1fbd2e2-7349-4491-b25b-6daca82f00d5-0' tool_calls=[{'name': 'retrieve', 'args': {'query': 'chats with anyone in March 2023'}, 'id': '29d99e36-cdf0-4c66-b28c-e68d1ca2c76d', 'type': 'tool_call'}] usage_metadata={'input_tokens': 3122, 'output_tokens': 29, 'total_tokens': 3151}
================================== Ai Message ==================================
Tool Calls:
  retrieve (29d99e36-cdf0-4c66-b28c-e68d1ca2c76d)
 Call ID: 29d99e36-cdf0-4c66-b28c-e68d1ca2c76d
  Args:

In [277]:
# participant and year
input_message = "conversations Damian 2024"
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

conversations Damian 2024
State messages content='' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b-instruct', 'created_at': '2025-08-07T20:05:11.895198Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3065470542, 'load_duration': 35895584, 'prompt_eval_count': 335, 'prompt_eval_duration': 1774381417, 'eval_count': 27, 'eval_duration': 1248659541, 'model_name': 'qwen2.5:7b-instruct'} id='run--7ffdeefe-9802-4b2f-8918-b68d8d5a92cd-0' tool_calls=[{'name': 'retrieve', 'args': {'query': 'conversations with Damian in 2024'}, 'id': 'be30e73f-df7b-4b73-a13d-95dbcb4917d7', 'type': 'tool_call'}] usage_metadata={'input_tokens': 335, 'output_tokens': 27, 'total_tokens': 362}
================================== Ai Message ==================================
Tool Calls:
  retrieve (be30e73f-df7b-4b73-a13d-95dbcb4917d7)
 Call ID: be30e73f-df7b-4b73-a13d-95dbcb4917d7
  Args:
    query: conversations w